# Olist Data Quality — Notebook educativo

## Fase 1: Preparación del entorno y reconocimiento de las fuentes

**Objetivo:** verificar Python, DuckDB y los archivos de Olist sin alterar CSV. Este notebook será ampliado de acuerdo con el orden del proyecto; **aquí no se implementan aún la extracción completa a RAW ni las 15 reglas**.

Consulta la guía `docs/FASE_1_TUTORIAL.md` para ejecutar los pasos desde Windows 11.

### Arquitectura fija

```text
CSV originales -> Extracción Python + DuckDB -> RAW
              -> 15 reglas SQL -> CLEAN
              -> pruebas críticas -> Parquet + reporte de calidad
```

RAW y CLEAN son las únicas capas persistentes de datos. `audit` será exclusivamente un esquema de control, no una capa STAGING.

### 1. Detectar la carpeta raíz y mostrar el entorno

En VS Code, selecciona **Python (olist-quality)** arriba a la derecha. El bloque siguiente localiza la raíz incluso si el directorio de trabajo es `notebooks/`.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((candidate for candidate in (cwd, *cwd.parents)
                     if (candidate / "environment.yml").exists()), None)
if project_root is None:
    raise RuntimeError("Abre el notebook desde la carpeta olist-data-quality")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Raíz:", project_root)
print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)

Raíz: C:\Users\Manuel Alexander\Documents\MIICC\A1\Trimestre 4\Introducción al Análisis de Datos\Proyectos\Proyecto 1\olist-data-quality
Python: 3.12.16
Intérprete: c:\Users\Manuel Alexander\.conda\envs\olist-quality\python.exe


### 2. Comprobar bibliotecas

Verificamos las versiones realmente instaladas. Si una importación falla, vuelve al Paso 4 del tutorial.

In [2]:
import duckdb
import pandas as pd
import pyarrow as pa
import matplotlib
print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("Matplotlib:", matplotlib.__version__)

DuckDB: 2.0.0
Pandas: 2.3.3
PyArrow: 24.0.0
Matplotlib: 3.11.2


In [3]:
import duckdb
from src.config import DB_PATH

con = duckdb.connect(str(DB_PATH))
try:
    sql = """SELECT schema_name FROM information_schema.schemata
             WHERE schema_name IN ('raw', 'clean', 'audit')
             ORDER BY schema_name"""
    print(con.execute(sql).fetchall())
finally:
    con.close()

[('audit',), ('clean',), ('raw',)]


### 3. Crear la base persistente

El módulo `src.bootstrap` es **la única implementación** de esta operación; el notebook sólo lo invoca. Repetir la celda no borra datos.

In [4]:
from src.bootstrap import bootstrap
from src.config import DB_PATH

print("Base de datos:", bootstrap())
connection = duckdb.connect(str(DB_PATH))
try:
    schema_rows = connection.execute("""
        SELECT schema_name FROM information_schema.schemata
        WHERE schema_name IN ('raw', 'clean', 'audit') ORDER BY schema_name
    """).fetchall()
    print("Esquemas:", schema_rows)
    assert {'raw', 'clean', 'audit'}.issubset({row[0] for row in schema_rows})
finally:
    connection.close()

Base de datos: C:\Users\Manuel Alexander\Documents\MIICC\A1\Trimestre 4\Introducción al Análisis de Datos\Proyectos\Proyecto 1\olist-data-quality\database\ecommerce.duckdb
Esquemas: [('audit',), ('clean',), ('raw',)]


### 4. Verificar archivos originales

Los nueve archivos oficiales deben copiarse a `data/raw/` sin cambios. Esta celda funciona antes de descargarlos: reporta los faltantes y no inventa cantidades.

In [5]:
from src.config import DATA_RAW_DIR, EXPECTED_CSV_FILES

present = sorted(path.name for path in DATA_RAW_DIR.glob('*.csv'))
missing = sorted(set(EXPECTED_CSV_FILES) - set(present))
print("CSV encontrados:", len(present))
print("CSV faltantes:", missing)
if missing:
    print("Descarga el dataset Olist siguiendo el Paso 7 del tutorial.")

CSV encontrados: 9
CSV faltantes: []


### 5. Inspeccionar el origen y generar reportes

**Sólo cuando estén los nueve archivos** se ejecuta el inventario completo. Se cuentan registros CSV (no líneas físicas) y se registra una huella SHA-256 que ayuda a verificar inmutabilidad. Los archivos originales no se modifican.

In [6]:
from src.source_inventory import build_inventory

if missing:
    print("Inventario pendiente: primero descargar los 9 CSV originales.")
else:
    inventory = build_inventory()
    inventory_df = pd.DataFrame(inventory)
    display(inventory_df[["file", "records_count", "columns_count", "row_width_mismatches"]])
    print("Inventario JSON y CSV creado en reports/quality_reports/")

,file,records_count,columns_count,row_width_mismatches
0,olist_customers_dataset.csv,99441,5,0
1,olist_geolocation_dataset.csv,1000163,5,0
2,olist_order_items_dataset.csv,112650,7,0
3,olist_order_payments_dataset.csv,103886,5,0
4,olist_order_reviews_dataset.csv,99224,7,0
5,olist_orders_dataset.csv,99441,8,0
6,olist_products_dataset.csv,32951,9,0
7,olist_sellers_dataset.csv,3095,4,0
8,product_category_name_translation.csv,71,2,0


Inventario JSON y CSV creado en reports/quality_reports/


### 6. Comprobar integridad de estructura sin convertir tipos

Los errores de ancho en un CSV se reportan para su investigación. Una fila irregular **no implica automáticamente** que deba eliminarse.

In [7]:
if missing:
    print("Validación del ancho pendiente: faltan archivos originales.")
else:
    width_issues = inventory_df.loc[inventory_df["row_width_mismatches"] > 0,
                                     ["file", "row_width_mismatches"]]
    if width_issues.empty:
        print("No se observaron filas con cantidad de campos distinta al encabezado.")
    else:
        print("Archivos con anomalías estructurales:")
        display(width_issues)

No se observaron filas con cantidad de campos distinta al encabezado.


### 7. Final de Fase 1

Una Fase 1 verificada tendrá: entorno Conda, kernel correcto, `database/ecommerce.duckdb`, nueve archivos Olist en `data/raw/` y reportes de inventario. **Aún no se han creado tablas RAW, CLEAN ni Parquet.**

La Fase 2 usará las columnas reales para determinar PK, FK, claves de negocio, granularidad y relaciones sin suponer que todos los duplicados o nulos son errores.